# LAB-15 · Fused softmax

Softmax turns scores into shares that sum to one, like dividing a pie according to everyone’s bids. You will normalize each row in one program, then keep a running maximum and total across smaller tiles so large scores do not overflow.

~25 minutes · **GPU optional**. A kernel is a small function run over many data items; Triton is a Python-based language for writing it. Run cells in order, fill the `# TODO` gaps, then use the checks; starter failures are expected.

Use a fresh Python runtime. Setup can install Triton in your account; it needs no API key or repository files. CPU correctness uses Triton's interpreter on a supported Linux runtime (Colab/Kaggle); native macOS is not a supported Triton runtime.


In [ ]:
import os
import sys
import torch

HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
# The interpreter setting must be chosen BEFORE the first import of Triton.
if "triton" in sys.modules:
    raise RuntimeError("Restart the runtime, then run this setup cell first (Triton is already imported).")
if HAS_GPU:
    os.environ.pop("TRITON_INTERPRET", None)
else:
    os.environ["TRITON_INTERPRET"] = "1"
    print("No GPU: correctness checks only, timings skipped")
try:
    import triton
except ImportError:
    # Runs only in your notebook runtime; PyTorch is preinstalled in Colab/Kaggle.
    %pip install -q triton
    import triton
import triton.language as tl

GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "none (CPU interpreter)"
print(f"torch {torch.__version__}; triton {triton.__version__}; GPU: {GPU_NAME}")
torch.manual_seed(0)


In [ ]:
# Self-check failures are reported without stopping Run all, so you can reach Solution.
# Solution checks are strict: a broken reference stops execution before benchmarking.
def check(label, test, strict=False):
    try:
        test()
    except Exception as error:
        print(f"FAIL {label}: {type(error).__name__}: {error}")
        if strict:
            raise
        return False
    print(f"PASS {label}")
    return True

# Vendor datasheet peaks, decimal GB/s. Match variants conservatively.
PEAK_GB_S = {"T4": 320.0, "P100 16 GB": 732.0, "H100 SXM": 3350.0}
def detected_peak():
    if not HAS_GPU:
        return None
    name = GPU_NAME.upper()
    if name in ("TESLA T4", "NVIDIA T4", "T4"):
        return PEAK_GB_S["T4"]
    memory = torch.cuda.get_device_properties(0).total_memory
    if "P100" in name and 15e9 <= memory <= 18e9:
        return PEAK_GB_S["P100 16 GB"]
    if "H100" in name and "SXM" in name:
        return PEAK_GB_S["H100 SXM"]
    return None

def report(label, milliseconds, byte_count):
    gb_s = byte_count / (milliseconds * 1e6)
    text = f"your run | {label}: {milliseconds:.4f} ms, {byte_count:,} logical bytes, {gb_s:.2f} GB/s"
    peak = detected_peak()
    if peak is not None:
        text += f", {100 * gb_s / peak:.1f}% of vendor peak"
    print(text)

def bench(label, call, byte_count):
    if not HAS_GPU:
        print(f"Timings skipped: {label} (CPU correctness only)")
        return
    from triton.testing import do_bench
    milliseconds = do_bench(call, warmup=25, rep=100)
    report(label, milliseconds, byte_count)


## 1 · One row, one counter

Subtracting the largest score before exponentiating preserves the shares and prevents overflow for large positive scores. A reduction combines the loaded values into one maximum or total, like counting everyone’s bids at the same counter. Complete the numerator; padded slots load negative infinity so they contribute zero. This teaching version accepts finite float32 rows up to 4,096 columns.


In [ ]:
@triton.jit
def row_softmax_kernel(X, Out, COLS: tl.constexpr, STRIDE: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    x = tl.load(X + row * STRIDE + cols, mask=cols < COLS, other=-float("inf"))
    maximum = tl.max(x, axis=0)
    numerator = tl.full((BLOCK,), 1.0, tl.float32)  # TODO: stable exponentials
    total = tl.sum(numerator, axis=0)
    tl.store(Out + row * COLS + cols, numerator / total, mask=cols < COLS)

def row_softmax(x, out=None):
    assert x.ndim == 2 and 0 < x.shape[1] <= 4096 and x.stride(1) == 1
    if out is None:
        out = torch.empty(x.shape, device=x.device, dtype=x.dtype)
    if x.shape[0]:
        row_softmax_kernel[(x.shape[0],)](x, out, x.shape[1], x.stride(0), triton.next_power_of_2(x.shape[1]))
    return out


In [ ]:
def check_softmax(implementation):
    # Finite inputs only; include tails, singleton rows, extremes and increasing tile maxima.
    for rows, cols in ((0, 17), (1, 1), (3, 7), (2, 65), (4, 129)):
        random = torch.randn((rows, cols), device=DEVICE, dtype=torch.float32)
        increasing = torch.linspace(-1000, 1000, cols, device=DEVICE).repeat(rows, 1)
        tied = torch.full((rows, cols), -1000.0, device=DEVICE)
        # Padded rows test a row stride larger than the logical width.
        padded = torch.randn((rows, cols + 5), device=DEVICE)[:, :cols]
        for x in (random, increasing, tied, padded):
            actual = implementation(x)
            expected = torch.softmax(x, dim=1)
            torch.testing.assert_close(actual, expected, rtol=2e-4, atol=2e-5)
            torch.testing.assert_close(actual.sum(dim=1), torch.ones(rows, device=DEVICE), rtol=2e-4, atol=2e-5)

check("one-row softmax", lambda: check_softmax(row_softmax))


## 2 · Carry a running maximum and total

Like updating a running bill when the currency changes, a new maximum changes the scale of every older contribution. Rescale the old total by exp(old_max − new_max), then add the new tile’s exponentials. Complete that update; a second pass rereads the scores to write the final shares (two input reads and one output write). This is the softmax bookkeeping behind tiled attention, not an attention kernel; finite, nonempty rows avoid undefined all-negative-infinity cases.


In [ ]:
@triton.jit
def online_softmax_kernel(X, Out, COLS: tl.constexpr, STRIDE: tl.constexpr, TILE: tl.constexpr):
    row = tl.program_id(0)
    lanes = tl.arange(0, TILE)
    maximum = -float("inf")
    total = 0.0
    for start in range(0, tl.cdiv(COLS, TILE)):
        cols = start * TILE + lanes
        x = tl.load(X + row * STRIDE + cols, mask=cols < COLS, other=-float("inf"))
        new_maximum = tl.maximum(maximum, tl.max(x, axis=0))
        total = total + tl.sum(tl.exp(x - new_maximum), axis=0)  # TODO: rescale the old total
        maximum = new_maximum
    # Normalized outputs need the final total: reread the input, without a full exp tensor.
    for start in range(0, tl.cdiv(COLS, TILE)):
        cols = start * TILE + lanes
        x = tl.load(X + row * STRIDE + cols, mask=cols < COLS, other=-float("inf"))
        tl.store(Out + row * COLS + cols, tl.exp(x - maximum) / total, mask=cols < COLS)

def online_softmax(x, out=None):
    assert x.ndim == 2 and x.shape[1] > 0 and x.stride(1) == 1
    if out is None:
        out = torch.empty(x.shape, device=x.device, dtype=x.dtype)
    if x.shape[0]:
        online_softmax_kernel[(x.shape[0],)](x, out, x.shape[1], x.stride(0), 32)
    return out


In [ ]:
check("running maximum and total, including changing maxima", lambda: check_softmax(online_softmax))


## Try first

Try filling the TODOs and rerunning each exercise and its check before reading on. Run all intentionally reports starter failures, then installs the complete definitions below and checks them again. Only checked solutions are timed.


## Solution


In [ ]:
@triton.jit
def row_softmax_kernel(X, Out, COLS: tl.constexpr, STRIDE: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    x = tl.load(X + row * STRIDE + cols, mask=cols < COLS, other=-float("inf"))
    maximum = tl.max(x, axis=0)
    numerator = tl.exp(x - maximum)
    total = tl.sum(numerator, axis=0)
    tl.store(Out + row * COLS + cols, numerator / total, mask=cols < COLS)

def row_softmax(x, out=None):
    assert x.ndim == 2 and 0 < x.shape[1] <= 4096 and x.stride(1) == 1
    if out is None:
        out = torch.empty(x.shape, device=x.device, dtype=x.dtype)
    if x.shape[0]:
        row_softmax_kernel[(x.shape[0],)](x, out, x.shape[1], x.stride(0), triton.next_power_of_2(x.shape[1]))
    return out


In [ ]:
@triton.jit
def online_softmax_kernel(X, Out, COLS: tl.constexpr, STRIDE: tl.constexpr, TILE: tl.constexpr):
    row = tl.program_id(0)
    lanes = tl.arange(0, TILE)
    maximum = -float("inf")
    total = 0.0
    for start in range(0, tl.cdiv(COLS, TILE)):
        cols = start * TILE + lanes
        x = tl.load(X + row * STRIDE + cols, mask=cols < COLS, other=-float("inf"))
        new_maximum = tl.maximum(maximum, tl.max(x, axis=0))
        total = total * tl.exp(maximum - new_maximum) + tl.sum(tl.exp(x - new_maximum), axis=0)
        maximum = new_maximum
    # Normalized outputs need the final total: reread the input, without a full exp tensor.
    for start in range(0, tl.cdiv(COLS, TILE)):
        cols = start * TILE + lanes
        x = tl.load(X + row * STRIDE + cols, mask=cols < COLS, other=-float("inf"))
        tl.store(Out + row * COLS + cols, tl.exp(x - maximum) / total, mask=cols < COLS)

def online_softmax(x, out=None):
    assert x.ndim == 2 and x.shape[1] > 0 and x.stride(1) == 1
    if out is None:
        out = torch.empty(x.shape, device=x.device, dtype=x.dtype)
    if x.shape[0]:
        online_softmax_kernel[(x.shape[0],)](x, out, x.shape[1], x.stride(0), 32)
    return out


In [ ]:
check("solution: row softmax", lambda: check_softmax(row_softmax), strict=True)
check("solution: online softmax", lambda: check_softmax(online_softmax), strict=True)


The full-row kernel logically reads N values and writes N: **8N bytes** for float32. The online version reads the input twice and writes once: **12N bytes**. The explicit unfused reference below reads N for max, 2N for subtraction, 2N for exp, N for sum and 2N for division, plus two row-scalar writes and two reads: **32N + 16R bytes** for R rows. Caches can change actual memory traffic. The running version limits the temporary vector width at the cost of rereading input; neither version promises better speed for every shape.


## Measure on a GPU

Results below are **your run**, not a promised speedup. Byte counts are logical tensor reads/writes, excluding allocation, cache effects and instruction overhead; GB/s is effective bandwidth, not measured HBM traffic, and can exceed a datasheet peak when data is cached. Times depend on shape, GPU, compiler and runtime.

The small peak table uses vendor datasheets: T4 320 GB/s, P100 **16 GB** 732 GB/s, H100 **SXM** 3.35 TB/s. Ambiguous or unknown GPU variants show GB/s without a peak percentage. No timings run in CPU-interpreter mode.


In [ ]:
if HAS_GPU:
    rows, cols = 2048, 1024  # Illustrative shape.
    x = torch.randn((rows, cols), device=DEVICE)
    maximum = torch.empty((rows, 1), device=DEVICE)
    total = torch.empty_like(maximum)
    shifted = torch.empty_like(x)
    exponentials = torch.empty_like(x)
    out = torch.empty_like(x)
    def unfused_softmax():
        torch.amax(x, dim=1, keepdim=True, out=maximum)
        torch.sub(x, maximum, out=shifted)
        torch.exp(shifted, out=exponentials)
        torch.sum(exponentials, dim=1, keepdim=True, out=total)
        torch.div(exponentials, total, out=out)
        return out
    expected = torch.softmax(x, dim=1)
    for name, call, byte_count in (
        ("explicit five-op PyTorch softmax", unfused_softmax, 32 * x.numel() + 16 * rows),
        ("one-row Triton softmax", lambda: row_softmax(x, out), 8 * x.numel()),
        ("running-total Triton softmax, two reads", lambda: online_softmax(x, out), 12 * x.numel()),
    ):
        torch.testing.assert_close(call(), expected, rtol=2e-4, atol=2e-5)
        bench(name, call, byte_count)
else:
    print("Timings skipped (CPU correctness only)")


## Take it further

You built stable row normalization and running-max/total normalization across tiles. Real Flash Attention also maintains weighted value accumulators and avoids materializing the attention score matrix; this lab rereads an already stored score matrix. Backs I34 (fused softmax to Flash-style bookkeeping); G10/G12 introduce the same math visually.

[Academy LAB-15 page](https://tile-and-error-academy.vercel.app/#lab/lab-15) · [Tile & Error channel](https://www.youtube.com/@tileanderror)
